# Classifier-free guidance — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normal_pdf(x,mu,sigma):
    return np.exp(-0.5*((x-mu)/sigma)**2)/(sigma*np.sqrt(2*np.pi))
def softmax(a):
    a=np.asarray(a,dtype=float); e=np.exp(a-a.max()); return e/e.sum()
def mse(a,b): return float(np.mean((np.asarray(a)-np.asarray(b))**2))
print('setup ok')

## conditional-unconditional score mixing
DDPM sampling (10.12) supplies the denoising prediction, conditioning supplies text or labels, and vector arithmetic explains the extrapolation. This becomes a central knob in latent text-to-image systems (10.14) and ControlNet (10.15).

In [ ]:
# 1) guidance extrapolates conditional minus unconditional score
su=-0.4; sc=0.9; ws=np.linspace(0,5,80); sg=su+ws*(sc-su)
plt.figure(figsize=(5,3)); plt.plot(ws,sg); plt.xlabel('guidance scale'); plt.title('guided score grows with scale')
assert abs(su+3*(sc-su)-3.5)<1e-12

In [ ]:
# 2) w=0 is unconditional and w=1 is conditional
vals=[su+0*(sc-su), su+1*(sc-su), su+3*(sc-su)]
plt.figure(figsize=(4,3)); plt.bar(['w=0','w=1','w=3'],vals); plt.title('three guidance regimes')
assert vals[0]==su and vals[1]==sc

In [ ]:
# 3) too much guidance can overshoot a target region
x=0.; target=1.; path=[]
for w in [1,3,6,9]: path.append(x+0.2*(su+w*(sc-su)))
plt.figure(figsize=(5,3)); plt.plot([1,3,6,9],path,'o-'); plt.axhline(target,c='r'); plt.title('large guidance overshoots')
assert path[-1]>target

In [ ]:
# 4) dropout training gives both conditional and unconditional cases
mask=np.array([0,1,1,0,1]); plt.figure(figsize=(4,3)); plt.bar(range(len(mask)),mask); plt.title('conditioning dropout mask')
assert mask.sum()==3

In [ ]:
# 5) vector guidance changes direction in 2D
su=np.array([-.4,.1]); sc=np.array([.9,.6]); guided=su+2*(sc-su)
plt.figure(figsize=(4,4)); plt.quiver([0,0,0],[0,0,0],[su[0],sc[0],guided[0]],[su[1],sc[1],guided[1]],angles='xy',scale_units='xy',scale=1); plt.axis('equal'); plt.title('uncond, cond, guided vectors')
assert np.linalg.norm(guided)>np.linalg.norm(sc)